# OpenAIRE Marine Science benchmark

Benchmark Data Gatherer against the curated OpenAIRE scholarly graph of the European Marine Science
community — *A Novel Curated Scholarly Graph Connecting Textual and Data Publications* (JDIQ 2023,
[10.1145/3597310](https://doi.org/10.1145/3597310)), data on [Zenodo 7464120](https://zenodo.org/record/7464120) (CC-BY 4.0).

| graph | content |
|---|---|
| `curated_MES` | 4,047 pubs · 5,488 datasets · 22 software · pub→data edges with `semantics` and curation `status` (`Validated` = OpenAIRE link confirmed, `Added` = found by curators in the PDF) |
| `curated_MES_with_removed_edges` | same + `Removed` edges (OpenAIRE links rejected by curators → curated negatives) |
| `original_MES_community` | uncurated OpenAIRE snapshot |

1,383 edges also carry `mention_position` (DAS, acknowledgments, table caption, in-text pointer) and `citation_type` (formal / informal).

**Workflow** (this directory is the experiment root):
1. `data/` – raw Zenodo dump (git-ignored)
2. `gt/` – ground truth: positives, curated negatives, dataset hierarchy, AutoDDG pairs
3. `input/` – `pmcid` CSV for `k8s/k8s_processor.py` (pilot = papers resolvable to PMC)
4. `output/<config>/dataset_citations.csv` – Data Gatherer runs
5. `eval/` – metrics, FP/FN dumps, recall breakdowns

## 0 · Setup

In [ ]:
import json
import logging
import re
import sys
import time
import zipfile
from collections import Counter
from pathlib import Path
from urllib.parse import unquote

import pandas as pd
import requests


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError(f"data-gatherer repo root not found above {start}")


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
ROOT = REPO_ROOT / "scripts" / "OpenAIRE"
DATA_DIR, GT_DIR, INPUT_DIR, OUTPUT_DIR, EVAL_DIR = (ROOT / d for d in ("data", "gt", "input", "output", "eval"))
for d in (DATA_DIR, GT_DIR, INPUT_DIR, OUTPUT_DIR, EVAL_DIR):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_ROOT))
from scripts.experiment_utils import evaluate_performance

ZENODO_URL = "https://zenodo.org/records/7464120/files/curated_MES_files.zip?download=1"
ZIP_PATH = DATA_DIR / "curated_MES_files.zip"
GRAPH_DIR = DATA_DIR / "curated_MES_files"

pd.set_option("display.max_colwidth", 120)
REPO_ROOT, ROOT

## 1 · Download the Zenodo dump

In [ ]:
if not GRAPH_DIR.exists():
    if not ZIP_PATH.exists():
        with requests.get(ZENODO_URL, stream=True, timeout=60) as r:
            r.raise_for_status()
            with open(ZIP_PATH, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)

sorted(p.relative_to(DATA_DIR) for p in GRAPH_DIR.glob("*/jsonl_files/*.jsonl"))

## 2 · Load graphs

In [ ]:
def load_graph(name: str) -> dict:
    d = GRAPH_DIR / name / "jsonl_files"
    return {k: pd.read_json(d / f"{k}.jsonl", lines=True, dtype=False)
            for k in ("publications", "datasets", "software", "relations")}


cur = load_graph("curated_MES")
rem = load_graph("curated_MES_with_removed_edges")

{k: len(v) for k, v in cur.items()}

In [ ]:
rel = cur["relations"]
rel[rel["semantics"] != "HasAuthor"].groupby(["semantics", "status"], dropna=False).size().unstack(fill_value=0)

## 3 · Canonical identifiers

Nodes carry lists of landing-page URLs. We derive:
- **publications** → DOI (most frequent DOI among URLs) and PMCID (from Europe PMC / PMC URLs, then NCBI ID converter on the DOI)
- **datasets / software** → DOI (PANGAEA `10.1594`, figshare `10.6084`, Zenodo `10.5281`, …), falling back to the first URL

In [ ]:
DOI_RE = re.compile(r"(10\.\d{4,9}/[^\s?#&]+)", re.I)
PMC_RE = re.compile(r"\b(PMC\d+)\b", re.I)


def _urls(v):
    return v if isinstance(v, list) else ([] if v is None or (isinstance(v, float) and pd.isna(v)) else [v])


def norm_doi(s: str) -> str:
    return unquote(s).strip().lower().rstrip("/.").removesuffix(".pdf")


def best_doi(urls):
    dois = Counter(norm_doi(m.group(1)) for u in _urls(urls) if (m := DOI_RE.search(str(u))))
    return dois.most_common(1)[0][0] if dois else None


def first_pmcid(urls):
    for u in _urls(urls):
        if m := PMC_RE.search(str(u)):
            return m.group(1).upper()
    return None


pubs = cur["publications"].copy()
pubs["title"] = pubs["title"].str[0]
pubs["abstract"] = pubs["description"].map(lambda d: " ".join(d) if isinstance(d, list) else d)
pubs["doi"] = pubs["url"].map(best_doi)
pubs["pmcid"] = pubs["url"].map(first_pmcid)

artifacts = pd.concat([cur["datasets"].assign(artifact_type="dataset"),
                       cur["software"].assign(artifact_type="software")], ignore_index=True)
artifacts["title"] = artifacts["title"].str[0]
artifacts["description_text"] = artifacts["description"].map(lambda d: " ".join(d) if isinstance(d, list) else d)
artifacts["doi"] = artifacts["url"].map(best_doi)
artifacts["identifier"] = artifacts["doi"].fillna(artifacts["url"].map(lambda u: (_urls(u) or [None])[0]))
artifacts["doi_prefix"] = artifacts["doi"].str.extract(r"^(10\.\d+)/", expand=False)

print(f"pubs with DOI: {pubs['doi'].notna().sum()}/{len(pubs)} | PMCID from URLs: {pubs['pmcid'].notna().sum()}")
print(f"artifacts with DOI: {artifacts['doi'].notna().sum()}/{len(artifacts)}")
artifacts["doi_prefix"].value_counts().head(10)

In [ ]:
# Resolve remaining PMCIDs through the NCBI ID converter (cached to gt/doi_to_pmcid.json).
# Only DOIs NCBI actually answered for are cached, so rate-limited batches are retried on re-run.
IDCONV_URL = "https://www.ncbi.nlm.nih.gov/pmc/utils/idconv/v1.0/"


def doi_to_pmcid(dois, batch_size=100, max_retries=5):
    out = {}
    for i in range(0, len(dois), batch_size):
        batch = dois[i:i + batch_size]
        for attempt in range(max_retries):
            r = requests.get(IDCONV_URL, params={"ids": ",".join(batch), "format": "json",
                                                 "tool": "data-gatherer", "email": "data-gatherer@nyu.edu"}, timeout=60)
            if r.status_code != 429:
                break
            time.sleep(2 ** attempt)
        if r.status_code != 200:
            print(f"batch {i}: HTTP {r.status_code}, will retry on next run")
            continue
        for rec in r.json().get("records", []):
            if rec.get("doi"):
                out[norm_doi(rec["doi"])] = rec.get("pmcid")
        time.sleep(0.4)
    return out


cache_path = GT_DIR / "doi_to_pmcid.json"
doi2pmc = json.loads(cache_path.read_text()) if cache_path.exists() else {}
todo = sorted(set(pubs.loc[pubs["pmcid"].isna() & pubs["doi"].notna(), "doi"]) - set(doi2pmc))
if todo:
    doi2pmc.update(doi_to_pmcid(todo))
    cache_path.write_text(json.dumps(doi2pmc, indent=1))

print(f"pubs with PMCID: {pubs['pmcid'].notna().sum()}/{len(pubs)}")

## 4 · Ground truth

One row per publication → artifact edge. Relations are stored in either direction
(`pub IsSupplementedBy ds`, `ds IsReferencedBy pub`, …), so we orient them to `pub_id → artifact_id`
and keep the original `semantics` plus `direction`.

`IsPartOf` edges between datasets give the PANGAEA collection hierarchy → `parent_identifier`, used for lenient matching.

In [ ]:
PUB_IDS = set(pubs["id"])
ART_IDS = set(artifacts["id"])


def pub_artifact_edges(relations: pd.DataFrame) -> pd.DataFrame:
    r = relations[relations["semantics"] != "HasAuthor"].copy()
    fwd = r["source"].isin(PUB_IDS) & r["target"].isin(ART_IDS)
    bwd = r["target"].isin(PUB_IDS) & r["source"].isin(ART_IDS)
    r = r[fwd | bwd].copy()
    r["direction"] = fwd[fwd | bwd].map({True: "pub→artifact", False: "artifact→pub"})
    r["pub_id"] = r["source"].where(r["direction"] == "pub→artifact", r["target"])
    r["artifact_id"] = r["target"].where(r["direction"] == "pub→artifact", r["source"])
    for col in ("mention_position", "mentioned_element"):
        r[col] = r[col].map(lambda v: "|".join(v) if isinstance(v, list) and v else None) if col in r else None
    if "citation_type" not in r:
        r["citation_type"] = None
    return r[["pub_id", "artifact_id", "semantics", "direction", "status",
              "citation_type", "mention_position", "mentioned_element"]]


# Dataset hierarchy: child IsPartOf parent
part_of = rel[(rel["semantics"] == "IsPartOf") & rel["source"].isin(ART_IDS) & rel["target"].isin(ART_IDS)]
id2ident = artifacts.set_index("id")["identifier"]
parent_of = part_of.groupby("source")["target"].first().map(id2ident)


def enrich(edges: pd.DataFrame) -> pd.DataFrame:
    out = (edges
           .merge(pubs[["id", "doi", "pmcid", "title"]].rename(columns={"id": "pub_id", "doi": "pub_doi", "title": "pub_title"}), on="pub_id")
           .merge(artifacts[["id", "artifact_type", "identifier", "doi_prefix", "title"]]
                  .rename(columns={"id": "artifact_id", "title": "artifact_title"}), on="artifact_id"))
    out["parent_identifier"] = out["artifact_id"].map(parent_of)
    return out.drop_duplicates(["pub_id", "artifact_id", "semantics"])


gt_all = enrich(pub_artifact_edges(rel))
neg = enrich(pub_artifact_edges(rem["relations"]).query("status == 'Removed'"))
# A pub/artifact pair can be removed under one semantics and kept under another: keep only true negatives
neg = neg[~neg.set_index(["pub_id", "artifact_id"]).index.isin(gt_all.set_index(["pub_id", "artifact_id"]).index)]

print(f"positive edges: {len(gt_all):,}  ({gt_all['pub_id'].nunique():,} pubs, {gt_all['artifact_id'].nunique():,} artifacts)")
print(f"negative edges: {len(neg):,}  ({neg['pub_id'].nunique():,} pubs)")
gt_all.groupby(["semantics", "status"]).size().unstack(fill_value=0)

In [ ]:
gt_all.to_parquet(GT_DIR / "marine_gt_edges.parquet", index=False)
neg.to_parquet(GT_DIR / "marine_gt_removed_edges.parquet", index=False)
pubs[["id", "doi", "pmcid", "title", "abstract", "url"]].to_parquet(GT_DIR / "marine_publications.parquet", index=False)
artifacts[["id", "artifact_type", "identifier", "doi", "doi_prefix", "title", "description_text", "url"]] \
    .to_parquet(GT_DIR / "marine_artifacts.parquet", index=False)
sorted(p.name for p in GT_DIR.iterdir())

## 5 · Pilot input (PMC subset)

`k8s/k8s_processor.py` takes a CSV with a `pmcid` column. The pilot covers every curated publication that
resolves to a PMCID — including those with **no** linked artifact, which test precision.

Data Gatherer is scored on **datasets** only (software edges are excluded from the GT below).

In [ ]:
pilot = pubs.loc[pubs["pmcid"].notna(), ["pmcid", "doi", "id"]].drop_duplicates("pmcid")
PILOT_NAME = "article_ids_marine_pmc.csv"
PILOT_CSV = INPUT_DIR / PILOT_NAME
pilot[["pmcid"]].to_csv(PILOT_CSV, index=False)
# k8s/run_loop.sh reads --input from k8s/input/<name>
pilot[["pmcid"]].to_csv(REPO_ROOT / "k8s" / "input" / PILOT_NAME, index=False)

pilot_ids = set(pilot["pmcid"])
has_ds = gt_all[(gt_all["artifact_type"] == "dataset") & gt_all["pmcid"].isin(pilot_ids)]
print(f"pilot articles: {len(pilot):,} | with ≥1 dataset: {has_ds['pmcid'].nunique():,} | dataset edges: {len(has_ds):,}")
has_ds["doi_prefix"].value_counts().head(8)

## 6 · Run Data Gatherer

Same CLIs as the BioDMS experiments (`scripts/BioDMS/test.ipynb`); every run writes `dataset_citations.csv`
under `output/`, and the eval in §7 picks up whatever exists.

**6a · Fine-tuned Flan-T5 on the HPC cluster** (`k8s/run_loop.sh`, `--no-enrich`) — free, rerun at will.
Configs mirror BioDMS: c2 semantic k=3 · c3 semantic k=3 + regex · c4 chunked full-document read · c5 regex only.
Each config uses its own `--job-suffix` (`-mc2`…`-mc5`) so S3 outputs don't collide with the BioDMS `-tc*` runs.

**6b · Claude Haiku 4.5 full-document read** (`k8s/k8s_processor.py`) — commented out (paid).

> ⚠️ `open_bio_data_repos.json` (the default ontology) has no PANGAEA / figshare / SeaDataNet entries.
> Expect DOI-style dataset mentions to rely on the regex/LLM path; a marine-extended ontology passed via
> `--seed-ontology` (T5) / `--ontology-path` (LLMs) is a natural follow-up config.

In [ ]:
import math

T5_MODEL = "hf-vida-nyu/flan-t5-base-dataref-info-extract"
SEED_ONTOLOGY = "data_gatherer/config/open_bio_data_repos.json"
N_PILOT = len(pilot)
N_GPUS = 4  # slices run in parallel; run_loop.sh times out each iteration at 2h
MAX_PER_SLICE = math.ceil(N_PILOT / N_GPUS)

# config label → dataset_citations.csv, relative to output/
CONFIGS = {
    "T5 · c3 (semantic k=3 + regex)":        "t5_c3/iter1/dataset_citations.csv",
    "T5 · c4 (chunked FDR)":                 "t5_c4/iter1/dataset_citations.csv",
    "Haiku · c4 (FDR)":                      "haiku_c4/dataset_citations.csv",
}
print(f"{N_PILOT} articles → {N_GPUS} slices × ≤{MAX_PER_SLICE}")

In [ ]:
# Flan-T5-finetuned -- S3 (c2)
!cd {REPO_ROOT} && bash k8s/run_loop.sh \
    --iterations 1 --gpus {N_GPUS} \
    --input {PILOT_NAME} \
    --max-articles-per-slice {MAX_PER_SLICE} \
    --output-dir {OUTPUT_DIR / "t5_c2"} \
    --seed-ontology {SEED_ONTOLOGY} \
    --model {T5_MODEL} \
    --job-suffix -mc2 \
    --semantic-retrieval true \
    --top-k 3 \
    --brute-force-regex false \
    --no-enrich

In [ ]:
# Flan-T5-finetuned -- RS3 (c3)
!cd {REPO_ROOT} && bash k8s/run_loop.sh \
    --iterations 1 --gpus {N_GPUS} \
    --input {PILOT_NAME} \
    --max-articles-per-slice {MAX_PER_SLICE} \
    --output-dir {OUTPUT_DIR / "t5_c3"} \
    --seed-ontology {SEED_ONTOLOGY} \
    --model {T5_MODEL} \
    --job-suffix -mc3 \
    --semantic-retrieval true \
    --top-k 3 \
    --brute-force-regex true \
    --no-enrich

In [ ]:
# Flan-T5-finetuned -- Full Document Chunk (c4)
!cd {REPO_ROOT} && bash k8s/run_loop.sh \
    --iterations 1 --gpus {N_GPUS} \
    --input {PILOT_NAME} \
    --max-articles-per-slice {MAX_PER_SLICE} \
    --output-dir {OUTPUT_DIR / "t5_c4"} \
    --seed-ontology {SEED_ONTOLOGY} \
    --model {T5_MODEL} \
    --job-suffix -mc4 \
    --top-k all \
    --semantic-retrieval false \
    --brute-force-regex false \
    --no-enrich

In [ ]:
# Flan-T5-finetuned -- R (c5)
!cd {REPO_ROOT} && bash k8s/run_loop.sh \
    --iterations 1 --gpus {N_GPUS} \
    --input {PILOT_NAME} \
    --max-articles-per-slice {MAX_PER_SLICE} \
    --output-dir {OUTPUT_DIR / "t5_c5"} \
    --seed-ontology {SEED_ONTOLOGY} \
    --model {T5_MODEL} \
    --job-suffix -mc5 \
    --semantic-retrieval false \
    --brute-force-regex true \
    --no-enrich

In [ ]:
# GPU energy per T5 config (sums all slices)
from scripts.experiment_utils import compute_gpu_energy_wh

for cfg in (p.split("/")[0] for p in CONFIGS.values() if p.startswith("t5_")):
    logs = sorted((OUTPUT_DIR / cfg / "iter1").glob("slice_*_gpu_power.csv"))
    if logs:
        print(cfg, [compute_gpu_energy_wh(str(p)) for p in logs])

In [ ]:
# # Claude Haiku 4.5 -- FDR
# !cd {REPO_ROOT} && python k8s/k8s_processor.py --input {PILOT_CSV} \
#     --output-dir {OUTPUT_DIR / "haiku_c4"} \
#     --model claude-haiku-4-5-20251001 --batch-size {N_PILOT} \
#     --full-document-read true --prompt-name CLAUDE_FDR_FewShot \
#     --use-batch-api false

In [ ]:
# # Collect async batch results — only if the FDR run above is switched to --use-batch-api true
# from data_gatherer.data_gatherer import DataGatherer
# dg = DataGatherer(log_level="INFO")
# batch_id, provider, cfg = "msgbatch_...", "anthropic", "haiku_c4"
# resp = OUTPUT_DIR / cfg / "batch_resp.jsonl"
# dg.parser.llm_client.download_batch_results(batch_id=batch_id, output_file_path=str(resp), api_provider=provider)
# dg.from_batch_resp_file_to_df(str(resp), output_file_path=str(OUTPUT_DIR / cfg / "dataset_citations.csv"))

## 7 · Evaluate

- **Article-level P/R/F1** with the shared `evaluate_performance` (macro-averaged, substring alias matching) — directly comparable to the BioDMS numbers.
- **Edge-level recall** broken down by curation `status` (`Added` = links OpenAIRE missed), `semantics`, repository and `mention_position`.
- **Lenient** recall: a prediction of the PANGAEA parent collection also counts.
- **Removed hits**: predicted links the curators rejected — FPs shared with the uncurated OpenAIRE graph.

In [ ]:
class _Orchestrator:
    """evaluate_performance only needs orchestrator.logger."""
    def __init__(self):
        self.logger = logging.getLogger("eval")


logging.basicConfig(level=logging.WARNING, format="%(message)s")
ORCH = _Orchestrator()

GT = gt_all[(gt_all["artifact_type"] == "dataset") & gt_all["pmcid"].isin(pilot_ids)].copy()
GT_EVAL = GT[["pmcid", "identifier"]].drop_duplicates()
GT_BASE = [f"https://www.ncbi.nlm.nih.gov/pmc/articles/{p}" for p in sorted(pilot_ids)]
NEG = neg[(neg["artifact_type"] == "dataset") & neg["pmcid"].isin(pilot_ids)]


def load_predictions(rel_path: str) -> pd.DataFrame | None:
    path = OUTPUT_DIR / rel_path
    if not path.exists():
        return None
    df = pd.read_csv(path)
    df = df[df["dataset_identifier"].notna()].copy()
    df["pmcid"] = df["source_url"].str.extract(r"(PMC\d+)", flags=re.I, expand=False).str.upper()
    df["pred"] = df["dataset_identifier"].astype(str).str.strip().str.lower().map(norm_doi)
    return df


def _alias(a: str, b) -> bool:
    return isinstance(b, str) and bool(b) and (a in b.lower() or b.lower() in a)


def mark_found(edges: pd.DataFrame, pred: pd.DataFrame, col: str = "identifier") -> pd.Series:
    by_pub = pred.groupby("pmcid")["pred"].apply(set).to_dict()
    return edges.apply(lambda r: any(_alias(p, r[col]) for p in by_pub.get(r["pmcid"], ())), axis=1)

In [ ]:
rows, edge_results = [], {}
for label, rel_path in CONFIGS.items():
    cfg = rel_path.split("/")[0]
    pred = load_predictions(rel_path)
    if pred is None:
        print(f"[skip] {label}: no output/{rel_path}")
        continue
    m = evaluate_performance(pred, GT_EVAL, ORCH,
                             str(EVAL_DIR / f"fp_{cfg}.txt"),
                             false_negatives_file=str(EVAL_DIR / f"fn_{cfg}.txt"),
                             repo_return=True, gt_base=GT_BASE)
    e = GT.copy()
    e["found"] = mark_found(e, pred)
    e["found_lenient"] = e["found"] | mark_found(e, pred, "parent_identifier")
    edge_results[cfg] = e
    removed_hits = mark_found(NEG, pred).sum() if len(NEG) else 0
    rows.append({"config": label, "articles": pred["pmcid"].nunique(), "predictions": len(pred),
                 "P": m["average_precision"], "R": m["average_recall"], "F1": m["f1_score"],
                 "edge_recall": e["found"].mean(), "edge_recall_lenient": e["found_lenient"].mean(),
                 "removed_hits": removed_hits})

summary = pd.DataFrame(rows)
if len(summary):
    summary.to_csv(EVAL_DIR / "summary.csv", index=False)
summary.round(3)

In [ ]:
def recall_by(col: str) -> pd.DataFrame:
    return pd.DataFrame({cfg: e.groupby(col, dropna=False)["found"].mean() for cfg, e in edge_results.items()}) \
             .assign(n=GT.groupby(col, dropna=False).size()).sort_values("n", ascending=False)


if edge_results:
    for col in ("status", "semantics", "doi_prefix", "mention_position", "citation_type"):
        display(recall_by(col).head(12).round(3))

## 8 · AutoDDG pairs

Dataset ↔ paper pairs for description-generation experiments. Papers that **supplement** or **document** a dataset
are the most informative sources. ~20% of dataset descriptions are near-copies of a linked paper's abstract —
flagged as `abstract_leak` so they can be excluded or analysed separately.

In [ ]:
from difflib import SequenceMatcher

LEAK_THRESHOLD = 0.8

pairs = (gt_all[gt_all["artifact_type"] == "dataset"]
         [["pub_id", "artifact_id", "semantics", "status", "identifier", "pub_doi", "pmcid"]]
         .merge(pubs[["id", "abstract"]].rename(columns={"id": "pub_id"}), on="pub_id")
         .merge(artifacts[["id", "title", "description_text"]].rename(columns={"id": "artifact_id", "title": "dataset_title"}),
                on="artifact_id"))


def _sim(a, b):
    if not isinstance(a, str) or not isinstance(b, str) or not a or not b:
        return None
    sm = SequenceMatcher(None, a[:1500], b[:1500], autojunk=False)
    # quick_ratio() is an upper bound on ratio(): skip the expensive call when it can't cross the threshold
    return sm.ratio() if sm.quick_ratio() > LEAK_THRESHOLD else sm.quick_ratio()


pairs["abstract_similarity"] = [_sim(a, b) for a, b in zip(pairs["abstract"], pairs["description_text"])]
pairs["abstract_leak"] = pairs["abstract_similarity"] > LEAK_THRESHOLD
pairs.to_parquet(GT_DIR / "autoddg_pairs.parquet", index=False)

print(f"pairs: {len(pairs):,} | datasets: {pairs['artifact_id'].nunique():,} | "
      f"datasets with a leaking abstract: {pairs.loc[pairs['abstract_leak'], 'artifact_id'].nunique():,}")
pairs.groupby("semantics")["abstract_leak"].agg(["size", "mean"]).round(3)